# Feature-set signatures {#sec-ind-feature-set-signatures}



## Preamble

### Introduction

Differential gene expression (DGE) analysis between cell subpopulations lets us
identify marker genes, i.e., sets of genes that distinguish one group of cells
from another. Clustering, however, will always give clusters, and DGE analysis
may or may not yield functionally relevant genes (e.g., phenotypic markers).

We can instead quantify sets of genes known to orchestrate biological functions 
or pathways (e.g., metabolic activity, cell cycle and death), both at level of
single cells and subpopulations (i.e., pseudo-bulk profiles). Approaches to do 
this rely on databases of transcription factor binding sites, gene regulatory 
networks, or annotated gene sets.

<!--
Complementarily, we can identify sets of features that, taken together, capture
the transcriptional variability of cells with regards to other variables (e.g., 
experimental condition); this is the goal of approaches such as MOFA. Here, the 
biological function of inferred factors is not known beforehand but needs to be 
investigated in downstream analyses.
-->

### Dependencies

In [ ]:
library(AUCell)
library(BiocParallel)
library(ggplot2)
library(ggspavis)
library(msigdbr)
library(patchwork)
library(pheatmap)
library(scrapper)
library(SpatialExperiment)
# parallelization setup
bp <- MulticoreParam(th <- 4)

In [ ]:
spe <- readRDS("img-spe_cl.rds")

## Set scoring

`r BiocStyle::Biocpkg("MSigDB")` provides a programmatic interface to the
[Molecular Signatures Database (MSigDB)](https://www.gsea-msigdb.org/gsea/msigdb)
[@Subramanian2005-MSigDB], one of the largest collections of molecular signatures and pathways.

Here, we query of human hallmark genes sets:

In [ ]:
# retrieve hallmark gene sets from 'MSigDB'
db <- msigdbr(species="Homo sapiens", collection="H")
gs <- split(db$ensembl_gene, db$gs_name)
# simplify gene set identifiers
names(gs) <- tolower(gsub("HALLMARK_", "", names(gs)))
# how many sets?
length(gs) 
# how many genes in each?
range(sapply(gs, length)) 

Next, we score these with `r BiocStyle::Biocpkg("AUCell")` [@Aibar2017-SCENIC],
a rank-based approach that (i) ranks genes for every observation (here, cells), 
and (ii) compute AUC values for each gene set. These represent the fraction of 
genes among top-ranked genes (default 5%) that are contained in a given set; 
i.e., values are in [0,1] and high values = high activity.

In [ ]:
# realize (sparse) gene expression matrix
mtx <- as(logcounts(spe), "dgCMatrix") 
# use ensembl identifiers as rownames
rownames(mtx) <- rowData(spe)$ID
# filter for genes represented in panel
.gs <- lapply(gs, intersect, rownames(mtx))
# keep only those with at least 5 genes
.gs <- .gs[sapply(.gs, length) >= 5]
# build per-cell gene rankings
rnk <- AUCell_buildRankings(mtx, BPPARAM=bp, plotStats=FALSE, verbose=FALSE)
# calculate AUC for each gene set in each cell
auc <- AUCell_calcAUC(geneSets=.gs, rankings=rnk, nCores=th, verbose=FALSE)
# add results as cell metadata
colData(spe)[rownames(auc)] <- t(assay(auc)) 

### Exploratory

Let's inspect the percentage of cells with non-zero score across signatures:

In [ ]:
fq <- rowMeans(assay(auc) > 0)
fq <- sort(round(100*fq, 2))
head(fq) # rarely detected
tail(fq) # mostly detected

We might also view how signature scores correlated with one another. Besides the 
underlying biology, this will be influenced by the number of genes overlapping 
between sets (here, we are only capturing a little over 300 RNA targets!).

In [ ]:
#| code-fold: true
cm <- cor(t(assay(auc)), method="spearman")
pheatmap(cm, 
    breaks=seq(-1, 1, 0.1), 
    color=pals::coolwarm(20), 
    cellwidth=10, cellheight=10)

Highly correlated sets will exhibit similar spatial patterns. Let's visualize 
some examples; here, hypoxia and myogenesis are spatially exclusive (they are
negatively correlated and occupy separated 'blocks' in the correlation matrix
displayed above), while apoptosis is ubiquitous (detected in >90% of cells).

In [ ]:
#| code-fold: true
lapply(c("hypoxia", "apoptosis", "myogenesis"), \(.) {
    q <- quantile(x <- spe[[.]], c(0.01, 0.99))
    x <- (x-q[1])/diff(q) # 01-quantile scaling
    x[x < 0] <- 0; x[x > 1] <- 1; spe[[.]] <- x
    plt <- plotCoords(spe, annotate=., point_size = 0.01) 
    plt
}) |> 
    wrap_plots(nrow=1, guides="collect") & 
    scale_color_gradientn(
        "q-scaled\nAUCell", 
        colors=hcl.colors(9, "Plasma")) &
    theme(
        legend.key.height=unit(1, "lines"),
        legend.key.width=unit(0.5, "lines"),
        panel.background=element_rect(fill="black")) 

Lastly, we can aggregate signature scores by cell subpopulations. This can help
us functionally characterize subpopulations that stem from, e.g., unsupervised 
approaches or in cases where the cells under study are poorly characterized.

In [ ]:
#| code-fold: true
# aggregate AUC values by cluster
pb <- aggregateAcrossCells.se(auc, colLabels(spe), assay.type="AUC")
mu <- sweep(assay(pb, "sums"), 2, pb$counts, `/`)
# visualize as (set x cluster) heatmap
pheatmap(mu, scale="row", col=pals::coolwarm())

In a multi-sample setting - say, where sections across different stages of
development, treatments, or health and disease - signatures may be compared 
between experimental conditions. Ideally, this'd be done at the subpopulation
level, since compositional differences are deemed to drive differences (e.g.,
sections richer in CD8+ T cells are, by chance, more likely to score higher
for cytotoxicity than sections where CD8+ T cells are absent, or similar).

## Appendix

### References {.unnumbered}